# Kazakh ASR Error Analysis — Whisper Small

Task 1: evaluate where ASR fails on Kazakh speech.

**Observed run:** 37 FLEURS Kazakh test utterances, 607.98 s (10.13 min). Whisper Small, no fine-tuning. Raw WER 80.25%, normalized WER 69.57%, CER 21.30%. Word errors: 368 substitutions, 64 deletions, 23 insertions.

Detailed CSV/JSON outputs are included in the submitted ZIP. This notebook is the reproducible pipeline.

In [ ]:
!pip -q install "transformers>=4.46,<5" "datasets>=3.0,<5" "accelerate>=1.0" "jiwer>=3.0" "soundfile>=0.12"

In [ ]:
import io, re, json, time, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, soundfile as sf, torch
from datasets import load_dataset, Audio
from transformers import pipeline
from jiwer import process_words, process_characters

MODEL_NAME="openai/whisper-small"
DATASET_NAME="google/fleurs"
DATASET_CONFIG="kk_kz"
SPLIT="test"
TARGET_SECONDS=600
RESULTS_DIR=Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
def decode_audio(audio_obj):
    src = io.BytesIO(audio_obj["bytes"]) if audio_obj.get("bytes") is not None else audio_obj["path"]
    wav, sr = sf.read(src, dtype="float32", always_2d=False)
    if wav.ndim == 2: wav = wav.mean(axis=1)
    return np.asarray(wav, dtype=np.float32), int(sr)

stream = load_dataset(DATASET_NAME, DATASET_CONFIG, split=SPLIT, streaming=True)
stream = stream.cast_column("audio", Audio(decode=False))
selected=[]; total=0.0
for idx, ex in enumerate(stream):
    wav, sr = decode_audio(ex["audio"])
    ref = str(ex.get("transcription", "")).strip()
    if not ref: continue
    dur=len(wav)/sr
    selected.append({"sample_id":f"fleurs_kk_{idx:04d}","audio":wav,"sampling_rate":sr,"duration_sec":dur,"reference_raw":ref})
    total += dur
    if total >= TARGET_SECONDS: break
assert total >= 600
print(len(selected), total/60)

In [ ]:
asr = pipeline("automatic-speech-recognition", model=MODEL_NAME, device=0 if torch.cuda.is_available() else -1, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
inputs=[{"raw":x["audio"],"sampling_rate":x["sampling_rate"]} for x in selected]
start=time.time()
outs=asr(inputs, batch_size=8, generate_kwargs={"language":"kazakh","task":"transcribe"})
elapsed=time.time()-start
for x,o in zip(selected, outs): x["prediction_raw"]=o["text"].strip()
print(f"Inference: {elapsed:.2f}s")

In [ ]:
def normalize_text(text):
    text=unicodedata.normalize("NFC", str(text)).lower()
    text="".join(" " if unicodedata.category(ch).startswith("P") else ch for ch in text)
    return re.sub(r"\s+", " ", text).strip()

refs_raw=[x["reference_raw"] for x in selected]
hyps_raw=[x["prediction_raw"] for x in selected]
refs=[normalize_text(x) for x in refs_raw]
hyps=[normalize_text(x) for x in hyps_raw]
raw=process_words(refs_raw, hyps_raw)
norm=process_words(refs, hyps)
chars=process_characters(refs, hyps)
metrics={
 "n_samples":len(selected), "audio_duration_sec":total, "audio_duration_min":total/60,
 "raw_wer":raw.wer, "normalized_wer":norm.wer, "normalized_cer":chars.cer,
 "word_substitutions":norm.substitutions, "word_deletions":norm.deletions, "word_insertions":norm.insertions, "word_hits":norm.hits,
 "inference_seconds":elapsed, "real_time_factor":elapsed/total, "model":MODEL_NAME, "dataset":f"{DATASET_NAME}/{DATASET_CONFIG}", "split":SPLIT
}
print(json.dumps(metrics, ensure_ascii=False, indent=2))

In [ ]:
rows=[]
for x,rn,hn in zip(selected,refs,hyps):
    w=process_words(rn,hn); c=process_characters(rn,hn)
    rows.append({"sample_id":x["sample_id"],"duration_sec":x["duration_sec"],"reference_raw":x["reference_raw"],"prediction_raw":x["prediction_raw"],"reference_norm":rn,"prediction_norm":hn,"wer":w.wer,"cer":c.cer})
predictions=pd.DataFrame(rows)
predictions.to_csv(RESULTS_DIR/"predictions.csv",index=False)
predictions[predictions.wer>0].to_csv(RESULTS_DIR/"errors.csv",index=False)
(RESULTS_DIR/"metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding="utf-8")
print(predictions[["sample_id","wer","reference_raw","prediction_raw"]].head(10).to_string(index=False))

In [ ]:
technical=pd.DataFrame([
 {"error_type":"substitution","count":norm.substitutions},
 {"error_type":"deletion","count":norm.deletions},
 {"error_type":"insertion","count":norm.insertions},
])
technical["share"]=technical["count"]/technical["count"].sum()
technical.to_csv(RESULTS_DIR/"technical_error_counts.csv",index=False)
print(technical.to_string(index=False))
print("\nWorst examples:")
print(predictions.sort_values(["wer","cer"],ascending=False).head(15)[["sample_id","wer","cer","reference_raw","prediction_raw"]].to_string(index=False))

## Interpretation

Substitutions dominated the observed word-level errors (368/455, about 80.9%). WER is much higher than CER, so many word tokens are counted wrong even when the character sequence remains closer. For Kazakh this can be important because suffix or character errors can invalidate a whole word under WER.

These figures should be trusted only for this 10.13-minute FLEURS subset. FLEURS is read/controlled speech and does not represent noisy calls, spontaneous speech, regional accents, or Kazakh–Russian code-switching.

If I had one month, I would first manually validate the dominant substitution groups on a larger and more diverse Kazakh evaluation set, then collect targeted data for the highest-impact verified failure mode before considering model adaptation.

**AI disclosure:** an AI assistant helped with experiment design, code review, error-analysis structure, and report formatting. All ASR predictions, metrics, tables, and quantitative conclusions come from the actual run.